In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import minimize, Bounds
import logging
import json
import os
import sys

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, WhiteKernel
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION (Module 23: Principal Directions) ---
INPUT_MASTER_FILE = 'bbo_master_w12.csv'
OUTPUT_DIR = 'add_data'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'week12_clean_inputs.json')

# Dimensionality Map [Source 156]
FUNCTION_DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}
XI = 0.01  
N_RESTARTS = 25

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s', stream=sys.stdout)

def expected_improvement(X, gpr, f_best, xi=0.01):
    """Calculates EI for Maximisation."""
    mu, sigma = gpr.predict(X.reshape(1, -1), return_std=True)
    with np.errstate(divide='ignore'):
        imp = mu - f_best - xi
        Z = imp / sigma
        ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0
    return -ei.flatten()

def generate_week12_queries():
    if not os.path.exists(INPUT_MASTER_FILE):
        logging.error(f"Input file {INPUT_MASTER_FILE} not found.")
        return

    df_master = pd.read_csv(INPUT_MASTER_FILE)
    final_queries = []

    for f_id in range(1, 9):
        dim = FUNCTION_DIMS[f_id]
        # FIX: Select only relevant X columns and drop rows with NaNs in X or Y [Source 165]
        x_cols = [f'X{i+1}' for i in range(dim)]
        df_f = df_master[df_master['Function ID'] == f_id].dropna(subset=x_cols + ['Y']).copy()
        
        X_raw = df_f[x_cols].values
        Y_raw = df_f['Y'].values.reshape(-1, 1)
        f_best = Y_raw.max()
        x_best = X_raw[np.argmax(Y_raw)]

        # --- STEP 1: Sensitivity Analysis (PCA Lens) ---
        scaler_x = StandardScaler()
        X_s = scaler_x.fit_transform(X_raw)
        
        # Matern with per-dimension length_scale to identify 'Principal Directions' [Source 17]
        kernel = C(1.0) * Matern(length_scale=np.ones(dim), nu=2.5) + WhiteKernel(noise_level=0.1)
        gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True)
        gpr.fit(X_s, Y_raw)
        
        # Identify low-impact dimensions via large length scales [Source 21]
        length_scales = gpr.kernel_.get_params()['k1__k2__length_scale']
        sensitivity = 1.0 / length_scales
        principal_dim_indices = np.argsort(sensitivity)[::-1] 

        # --- STEP 2: Dimensionality Reduction (Freezing) [Source 20] ---
        # Freeze 50% of dimensions for high-D functions to reduce complexity while retaining info
        num_to_freeze = dim // 2 if dim > 3 else 0
        frozen_indices = principal_dim_indices[-num_to_freeze:] if num_to_freeze > 0 else []
        active_indices = [i for i in range(dim) if i not in frozen_indices]

        logging.info(f"F{f_id} ({dim}D): Active dims = {active_indices}, Frozen dims = {frozen_indices}")

        def active_acq_wrapper(x_active):
            x_full = np.array(x_best, copy=True)
            for i, idx in enumerate(active_indices):
                x_full[idx] = x_active[i]
            x_s = scaler_x.transform(x_full.reshape(1, -1))
            return expected_improvement(x_s, gpr, f_best, xi=XI)

        best_val = np.inf
        next_x_active = None
        for _ in range(N_RESTARTS):
            x0 = np.random.uniform(0, 1, len(active_indices))
            res = minimize(active_acq_wrapper, x0, method='L-BFGS-B', bounds=[(0, 1)]*len(active_indices))
            if res.fun < best_val:
                best_val = res.fun
                next_x_active = res.x

        query_full = np.array(x_best, copy=True)
        for i, idx in enumerate(active_indices):
            query_full[idx] = next_x_active[i]

        q_clean = [float(round(x, 6)) for x in query_full]
        final_queries.append(q_clean)

    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    with open(OUTPUT_FILE, 'w') as f:
        json.dump(final_queries, f, indent=4)
    logging.info(f"SUCCESS: Generated queries saved to {OUTPUT_FILE}")

if __name__ == '__main__':
    generate_week12_queries()


INFO: F1 (2D): Active dims = [0, 1], Frozen dims = []
INFO: F2 (2D): Active dims = [0, 1], Frozen dims = []


/Users/larry/Applications/miniforge3/envs/jenv/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


INFO: F3 (3D): Active dims = [0, 1, 2], Frozen dims = []
INFO: F4 (4D): Active dims = [0, 1], Frozen dims = [2 3]


/Users/larry/Applications/miniforge3/envs/jenv/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


INFO: F5 (4D): Active dims = [0, 2], Frozen dims = [3 1]
INFO: F6 (5D): Active dims = [1, 2, 3], Frozen dims = [4 0]


...

INFO: F7 (6D): Active dims = [2, 4, 5], Frozen dims = [0 1 3]


/Users/larry/Applications/miniforge3/envs/jenv/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


INFO: F8 (8D): Active dims = [1, 2, 4, 7], Frozen dims = [6 5 3 0]


/Users/larry/Applications/miniforge3/envs/jenv/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/larry/Applications/miniforge3/envs/jenv/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/larry/Applications/miniforge3/envs/jenv/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a

INFO: SUCCESS: Generated queries saved to add_data/week12_clean_inputs.json
